# RAG POC for Engineering and Marketing Teams

Belinda Xu | UC Berkeley Master of Information and Data Science | December 1, 2026

A shared retrieval pipeline with audience-specific prompts for engineering and marketing teams. This project compares Cohere and Mistral, then tests chunk size, retrieval depth, and prompt changes using cosine similarity and ROUGE-L F1.

Developed for DATASCI 267 using a course-provided scenario and gold answers for engineering and marketing. The gold data contains 75 questions and answers. The small, random sample of 15 questions to experiment with different configurations is part of the full 75 question validation set.


## Setup


In [ ]:
%pip install -q -r requirements.txt

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import bs4
import requests
import torch
import wikipedia
from IPython.display import display
from rouge_score import rouge_scorer

os.environ.setdefault("USER_AGENT", "BelindaXu-RAG-Portfolio/1.0")

from langchain_community.document_loaders import PyMuPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_cohere import ChatCohere
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


In [ ]:
# read keys from the environment or Colab secrets
def get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return None
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None

COHERE_API_KEY = get_secret("COHERE_API_KEY")
hf_token = get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
if not COHERE_API_KEY:
    raise ValueError("Set COHERE_API_KEY in your environment or Colab secrets.")


In [ ]:
# load the course-provided questions and audience-specific gold answers
data_dir = Path("data")
validation_questions_answers = {
    int(key): value for key, value in json.loads(
        (data_dir / "gold_answers.json").read_text()).items()
}
test_questions = {
    int(key): value for key, value in json.loads(
        (data_dir / "additional_questions.json").read_text()).items()
}
assert len(validation_questions_answers) == 75
assert len(test_questions) == 30


## Embeddings and generation models


In [ ]:
base_embeddings = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")

In [ ]:
# load Mistral in 4-bit precision

quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
llm_mistral_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    dtype=torch.float32,
    device_map='auto',
    quantization_config=quantization_config
)

llm_mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

In [ ]:
mistral_pipe = pipeline(
    "text-generation",
    model=llm_mistral_model,
    tokenizer=llm_mistral_tokenizer,
    max_new_tokens=1000,
    temperature=0.6,
    top_p=0.95,
    do_sample=True,
    repetition_penalty=1.2
)
mistral_pipe.model.config.pad_token_id = mistral_pipe.model.config.eos_token_id

In [ ]:
mistral_llm_lc = HuggingFacePipeline(pipeline=mistral_pipe)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Document collection and baseline index


In [ ]:
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

documents = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=128, chunk_overlap=0)
splits = text_splitter.split_documents(documents)
print('Number of splits/chunks: ', str(len(splits)))

In [ ]:
qdrant_vectorstore = QdrantVectorStore.from_documents(splits,
    base_embeddings,
    location=":memory:",  # Local mode with in-memory storage only
    collection_name="rag_tech_db",
    force_recreate=True
)

retriever = qdrant_vectorstore.as_retriever()

In [ ]:
# track source documents
global_doc_number = 1

### Research papers


In [ ]:
arxiv_numbers = ('2005.11401', '2104.07567', '2104.09864', '2105.03011', '2106.09685', '2203.02155', '2203.15556', '2211.09260', '2211.12561',
                 '2212.09741', '2305.14314', '2305.18290', '2306.15595', '2309.08872', '2309.15217', '2310.06825', '2310.11511',
                 '2311.08377', '2312.05708', '2401.06532', '2401.17268', '2402.01306', '2402.19473', '2406.04744',
                 '2312.10997', '2410.12812', '2410.15944', '2411.16594', '2404.00657', '2601.07711', '2507.09477', '2506.10408',
                 '2605.05538', '2604.14220', '2606.12191', '2604.01733')


In [ ]:
all_arxiv_pages = []

# load research papers
for identifier in arxiv_numbers:

    arx_url = f"https://arxiv.org/pdf/{identifier}.pdf"

    arx_loader = PyMuPDFLoader(arx_url)
    arx_pages = arx_loader.load()
    for page_num in range(len(arx_pages)):
        page = arx_pages[page_num]

        page.metadata['page_num'] = page_num
        page.metadata['doc_num'] = global_doc_number
        page.metadata['doc_source'] = "ArXiv"
        all_arxiv_pages.append(page)

    global_doc_number += 1

In [ ]:
num_pages = len(all_arxiv_pages)
num_docs = global_doc_number - 1

print(f"{num_docs} documents in total")
print(f"{num_pages} pages in total")

36 documents in total
795 pages in total


In [ ]:
#index doc chunks
splits = text_splitter.split_documents(all_arxiv_pages)
for idx, text in enumerate(splits):
    splits[idx].metadata['split_id'] = idx

print('Number of splits/chunks: ', len(splits))

In [ ]:
qdrant_vectorstore.add_documents(documents=splits)

### Wikipedia


In [ ]:
import wikipedia
from langchain_core.documents import Document

# set a user agent for Wikipedia requests
wikipedia.set_user_agent("BelindaXu-RAG-Portfolio/1.0")

def robust_wikipedia_loader(query, load_max_docs=4):
    documents = []
    try:
        # Search for matching pages
        search_results = wikipedia.search(query, results=load_max_docs)
        print(f"Found Wikipedia search results for '{query}': {search_results}")

        for title in search_results:
            try:
                # space out requests
                time.sleep(0.5)
                page = wikipedia.page(title, auto_suggest=False)
                doc = Document(
                    page_content=page.content,
                    metadata={
                        "title": page.title,
                        "source": page.url
                    }
                )
                documents.append(doc)
            except Exception as inner_e:
                print(f"Skipping page '{title}' due to: {inner_e}")

    except Exception as e:
        print(f"Wikipedia search failed: {e}")
    return documents

# load and index Wikipedia pages
try:
    wiki_docs = robust_wikipedia_loader("Generative Artificial Intelligence", load_max_docs=4)

    # keep source metadata
    for idx, text in enumerate(wiki_docs):
        wiki_docs[idx].metadata['doc_num'] = global_doc_number
        wiki_docs[idx].metadata['doc_source'] = "Wikipedia"
        global_doc_number += 1

    print('Successfully fetched documents: ', len(wiki_docs))

    wiki_splits = text_splitter.split_documents(wiki_docs)
    for idx, text in enumerate(wiki_splits):
        wiki_splits[idx].metadata['split_id'] = idx

    print('Number of splits/chunks created: ', len(wiki_splits))
except Exception as e:
    print(f"Alternative approach also encountered an issue: {e}")
    wiki_splits = []

In [ ]:
# save wiki documents for experiments
all_wiki_docs = wiki_docs.copy()

In [ ]:
qdrant_vectorstore.add_documents(documents=wiki_splits)

In [ ]:
# load and index Wikipedia pages
try:
    wiki_docs = robust_wikipedia_loader("Information Retrieval", load_max_docs=2)

    # keep source metadata
    for idx, text in enumerate(wiki_docs):
        wiki_docs[idx].metadata['doc_num'] = global_doc_number
        wiki_docs[idx].metadata['doc_source'] = "Wikipedia"
        global_doc_number += 1

    print('Successfully fetched documents: ', len(wiki_docs))

    wiki_splits = text_splitter.split_documents(wiki_docs)
    for idx, text in enumerate(wiki_splits):
        wiki_splits[idx].metadata['split_id'] = idx

    print('Number of splits/chunks created: ', len(wiki_splits))
except Exception as e:
    print(f"Alternative approach also encountered an issue: {e}")
    wiki_splits = []

In [ ]:
# add information retrieval documents
all_wiki_docs.extend(wiki_docs)

In [ ]:
qdrant_vectorstore.add_documents(documents=wiki_splits)

In [ ]:
# load and index Wikipedia pages
try:
    wiki_docs = robust_wikipedia_loader("Large Language Models", load_max_docs=2)

    # keep source metadata
    for idx, text in enumerate(wiki_docs):
        wiki_docs[idx].metadata['doc_num'] = global_doc_number
        wiki_docs[idx].metadata['doc_source'] = "Wikipedia"
        global_doc_number += 1

    print('Successfully fetched documents: ', len(wiki_docs))

    wiki_splits = text_splitter.split_documents(wiki_docs)
    for idx, text in enumerate(wiki_splits):
        wiki_splits[idx].metadata['split_id'] = idx

    print('Number of splits/chunks created: ', len(wiki_splits))
except Exception as e:
    print(f"Alternative approach also encountered an issue: {e}")
    wiki_splits = []

In [ ]:
# add llm documents
all_wiki_docs.extend(wiki_docs)

In [ ]:
qdrant_vectorstore.add_documents(documents=wiki_splits)

In [ ]:
# load and index Wikipedia pages
try:
    wiki_docs = robust_wikipedia_loader("Retrieval Augmented Generation", load_max_docs=2)

    # keep source metadata
    for idx, text in enumerate(wiki_docs):
        wiki_docs[idx].metadata['doc_num'] = global_doc_number
        wiki_docs[idx].metadata['doc_source'] = "Wikipedia"
        global_doc_number += 1

    print('Successfully fetched documents: ', len(wiki_docs))

    wiki_splits = text_splitter.split_documents(wiki_docs)
    for idx, text in enumerate(wiki_splits):
        wiki_splits[idx].metadata['split_id'] = idx

    print('Number of splits/chunks created: ', len(wiki_splits))
except Exception as e:
    print(f"Alternative approach also encountered an issue: {e}")
    wiki_splits = []

In [ ]:
# add RAG documents
all_wiki_docs.extend(wiki_docs)

In [ ]:
qdrant_vectorstore.add_documents(documents=wiki_splits)

### Technical blogs


In [ ]:
# already loaded "https://lilianweng.github.io/posts/2023-06-23-agent/",
web_loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2020-10-29-odqa/",
               "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
               "https://lilianweng.github.io/posts/2018-06-24-attention/",
               "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
               "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
               "https://lilianweng.github.io/posts/2026-07-04-harness/",
               "https://huyenchip.com/2025/01/16/ai-engineering-pitfalls.html",
               "https://blog.langchain.com/the-rise-of-context-engineering/"),

    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

web_documents = web_loader.load()

for idx, text in enumerate(web_documents):
    web_documents[idx].metadata['doc_num'] = global_doc_number
    web_documents[idx].metadata['doc_source'] = "WWW"
    global_doc_number += 1

print('Number of web documents: ', len(web_documents))

In [ ]:
web_splits = text_splitter.split_documents(web_documents)

for idx, text in enumerate(web_splits):
    web_splits[idx].metadata['split_id'] = idx

print('Number of splits: ', len(web_splits))

In [ ]:
qdrant_vectorstore.add_documents(documents=web_splits)

### Business sources


In [ ]:
import requests
import bs4
from langchain_core.documents import Document
import time

web_paths = (
    "https://www.salesforce.com/ap/agentforce/what-is-rag",
    "https://www.meilisearch.com/blog/rag-for-business",
    "https://www.ibm.com/think/topics/retrieval-augmented-generation",
    "https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag",
    "https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation/"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

debugged_documents = []

for url in web_paths:
    print(f"Fetching: {url} ... ", end="", flush=True)
    start_time = time.time()
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = bs4.BeautifulSoup(response.text, "html.parser")

        # Try filtering by the blog classes first
        content_div = soup.find(class_=("post-content", "post-title", "post-header"))

        if content_div and content_div.get_text().strip():
            text_content = content_div.get_text()
        else:
            # fall back to body text
            for element in soup(["script", "style", "nav", "footer", "header"]):
                element.decompose()
            text_content = soup.get_text()

        # Clean up whitespace
        lines = (line.strip() for line in text_content.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        cleaned_text = "\n".join(chunk for chunk in chunks if chunk)

        doc = Document(
            page_content=cleaned_text,
            metadata={"source": url}
        )
        debugged_documents.append(doc)
        print(f"SUCCESS - Extracted {len(cleaned_text)} characters ({time.time() - start_time:.2f}s)")
    except Exception as e:
        print(f"FAILED with error: {e}")

In [ ]:
# index the fetched business sources
valid_web_docs = []
for doc in debugged_documents:
    if doc.page_content.strip():
        doc.metadata['doc_num'] = global_doc_number
        doc.metadata['doc_source'] = "WWW"
        global_doc_number += 1
        valid_web_docs.append(doc)

print(f"\nSuccessfully retrieved and prepared {len(valid_web_docs)} enterprise web documents.")

enterprise_splits = text_splitter.split_documents(valid_web_docs)
for idx, chunk in enumerate(enterprise_splits):
    chunk.metadata['split_id'] = idx

print(f"Created {len(enterprise_splits)} chunks from the enterprise web documents.")

if enterprise_splits:
    qdrant_vectorstore.add_documents(documents=enterprise_splits)
    print("Successfully ingested enterprise chunks into the Qdrant Vector Store!")
else:
    print("⚠️ No chunks were created. Please verify that the parsed text content wasn't empty.")

## Audience-specific RAG chains


In [ ]:
# create retriever to search documents in Qdrant
retriever = qdrant_vectorstore.as_retriever()

# connect to cohere
cohere_chat_model = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model="command-a-03-2025"
)

# create parser to extract answer text
output_parser = StrOutputParser()

In [ ]:
# engineering prompt
engineering_template = """
Answer below question for an engineering audience.
Give a detailed technical explanation using the context below.
Context: {context}
Question: {question}
"""

engineering_prompt = ChatPromptTemplate.from_template(engineering_template)

In [ ]:
# marketing prompt
marketing_template = """
Answer below question for a marketing audience.
Give a clear, high-level explanation using the context below. Use simple and concsie language.
Context: {context}
Question: {question}
"""

marketing_prompt = ChatPromptTemplate.from_template(marketing_template)

In [ ]:
# build engineering chain
engineering_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | engineering_prompt
    | cohere_chat_model
    | output_parser
)

# build marketing chain
marketing_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | marketing_prompt
    | cohere_chat_model
    | output_parser
)

In [ ]:
# create retriever to search documents in Qdrant
retriever = qdrant_vectorstore.as_retriever()

# use Mistral
mistral_rag_model = mistral_llm_lc.bind(
    pipeline_kwargs={"return_full_text": False}
)

# create parser to extract answer text
output_parser = StrOutputParser()

In [ ]:
# engineering prompt
mistral_engineering_template = """[INST]
Answer below question for an engineering audience.
Give a detailed technical explanation using the context below.
Context: {context}
Question: {question}
[/INST]"""

mistral_engineering_prompt = PromptTemplate.from_template(mistral_engineering_template)

In [ ]:
# marketing prompt
mistral_marketing_template = """[INST]
Answer below question for a marketing audience.
Give a clear, high-level explanation using the context below. Use simple and concise language.
Context: {context}
Question: {question}
[/INST]"""

mistral_marketing_prompt = PromptTemplate.from_template(mistral_marketing_template)

In [ ]:
# build engineering chain
mistral_engineering_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | mistral_engineering_prompt
    | mistral_rag_model
    | output_parser)

# build marketing chain
mistral_marketing_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | mistral_marketing_prompt
    | mistral_rag_model
    | output_parser)

## Evaluation metrics


In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

In [ ]:
# compare answer embeddings
def cos_sim(a, b):

    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

## Baseline comparison: 15 questions


In [ ]:
import random
random.seed(1234)

# get validation question ids and randomly select 15
question_ids = list(validation_questions_answers.keys())
rand_15 = random.sample(question_ids, 15)

print("Random 15x question IDs: ", rand_15)

Random 15x question IDs:  [161, 38, 0, 28, 13, 27, 33, 139, 108, 5, 11, 193, 83, 133, 129]


In [ ]:
cohere_eng_results = []

for question_id in rand_15:

    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_research"]

    answer = engineering_chain.invoke(question)

    # turn both answers into embeddings and compare cos similarity
    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    # compare matching words w rouge scores, pull f1 out
    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_eng_results.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

In [ ]:
cohere_mkt_results = []

for question_id in rand_15:

    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_marketing"]

    answer = marketing_chain.invoke(question)

    # turn both answers into embeddings and compare cos similarity
    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    # compare matching words w rouge scores, pull f1 out
    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_mkt_results.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

In [ ]:
mistral_eng_results = []

for question_id in rand_15:

    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_research"]

    answer = mistral_engineering_chain.invoke(question)

    # turn both answers into embeddings and compare cos similarity
    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    # compare matching words w rouge scores, pull f1 out
    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    mistral_eng_results.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

In [ ]:
mistral_mkt_results = []

for question_id in rand_15:

    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_marketing"]

    answer = mistral_marketing_chain.invoke(question)

    # turn both answers into embeddings and compare cos similarity
    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    # compare matching words w rouge scores, pull f1 out
    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    mistral_mkt_results.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

In [ ]:
cohere_eng_df = pd.DataFrame(cohere_eng_results)
cohere_mkt_df = pd.DataFrame(cohere_mkt_results)
mistral_eng_df = pd.DataFrame(mistral_eng_results)
mistral_mkt_df = pd.DataFrame(mistral_mkt_results)

In [ ]:
# avg scores for each model and team
baseline_comparison = pd.DataFrame(
    [
        ["Cohere", "Engineering",
         cohere_eng_df["cos similarity"].mean(),
         cohere_eng_df["ROUGE-L F1"].mean()],

        ["Mistral", "Engineering",
         mistral_eng_df["cos similarity"].mean(),
         mistral_eng_df["ROUGE-L F1"].mean()],

        ["Cohere", "Marketing",
         cohere_mkt_df["cos similarity"].mean(),
         cohere_mkt_df["ROUGE-L F1"].mean()],

        ["Mistral", "Marketing",
         mistral_mkt_df["cos similarity"].mean(),
         mistral_mkt_df["ROUGE-L F1"].mean()]
    ],
    columns=["Model", "Team", "Avg cos similarity", "Avg ROUGE-L F1"]
)

baseline_comparison

,Model,Team,Avg cos similarity,Avg ROUGE-L F1
0,Cohere,Engineering,0.792634,0.138782
1,Mistral,Engineering,0.781594,0.165601
2,Cohere,Marketing,0.804095,0.226675
3,Mistral,Marketing,0.806592,0.224068


## Experiment 1: 1,000-character chunks


In [ ]:
# reuse already loaded documents for experiment 1
experiment_documents_1000 = (
    documents + all_arxiv_pages + all_wiki_docs
    + web_documents + valid_web_docs)

# tweak original splitter to 1000 characters and keep overlap at 0
text_splitter_1000 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# split into larger chunks
chunks_1000 = text_splitter_1000.split_documents(experiment_documents_1000)

# index the larger chunks
qdrant_vectorstore_1000 = QdrantVectorStore.from_documents(
    chunks_1000,
    base_embeddings,
    location=":memory:",
    collection_name="rag_tech_db_1000")

# search new store and keep retrieval at 4 chunks
retriever_1000 = qdrant_vectorstore_1000.as_retriever(search_kwargs={"k": 4})

# reuse engineering prompt and model with new retriever
engineering_chain_1000 = (
    {"context": retriever_1000 | format_docs,
     "question": RunnablePassthrough()}
    | engineering_prompt
    | cohere_chat_model
    | output_parser)

# reuse marketing prompt and model with new retriever
marketing_chain_1000 = (
    {"context": retriever_1000 | format_docs,
     "question": RunnablePassthrough()}
    | marketing_prompt
    | cohere_chat_model
    | output_parser)

# evaluate engineering answers
cohere_eng_results_1000 = []

for question_id in rand_15:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_research"]

    answer = engineering_chain_1000.invoke(question)

    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_eng_results_1000.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

# evaluate marketing answers
cohere_mkt_results_1000 = []

for question_id in rand_15:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_marketing"]

    answer = marketing_chain_1000.invoke(question)

    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_mkt_results_1000.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

# collect scores
cohere_eng_df_1000 = pd.DataFrame(cohere_eng_results_1000)
cohere_mkt_df_1000 = pd.DataFrame(cohere_mkt_results_1000)

# average scores by audience
comparison_1000 = pd.DataFrame(
    [
        ["Cohere", "Engineering",
         cohere_eng_df_1000["cos similarity"].mean(),
         cohere_eng_df_1000["ROUGE-L F1"].mean()],

        ["Cohere", "Marketing",
         cohere_mkt_df_1000["cos similarity"].mean(),
         cohere_mkt_df_1000["ROUGE-L F1"].mean()]
    ],
    columns=["Model", "Team", "Avg cos similarity", "Avg ROUGE-L F1"]
)

# compare configurations
experiment_comparison = pd.concat(
    [baseline_comparison.assign(Configuration="Baseline: 128 characters"),
     comparison_1000.assign(Configuration="Experiment 1: 1000 characters")],
    ignore_index=True)

experiment_comparison

,Model,Team,Avg cos similarity,Avg ROUGE-L F1,Configuration
0,Cohere,Engineering,0.792634,0.138782,Baseline: 128 characters
1,Mistral,Engineering,0.781594,0.165601,Baseline: 128 characters
2,Cohere,Marketing,0.804095,0.226675,Baseline: 128 characters
3,Mistral,Marketing,0.806592,0.224068,Baseline: 128 characters
4,Cohere,Engineering,0.800110,0.138352,Experiment 1: 1000 characters
5,Cohere,Marketing,0.810163,0.200146,Experiment 1: 1000 characters


## Experiment 2: eight retrieved chunks


In [ ]:
# reuse experiment 1 vector store and retrieve 8 chunks instead of 4
retriever_1000_k8 = qdrant_vectorstore_1000.as_retriever(search_kwargs={"k": 8})

# reuse engineering prompt and model with new retriever
engineering_chain_1000_k8 = (
    {"context": retriever_1000_k8 | format_docs,
     "question": RunnablePassthrough()}
    | engineering_prompt
    | cohere_chat_model
    | output_parser)

# reuse marketing prompt and model with new retriever
marketing_chain_1000_k8 = (
    {"context": retriever_1000_k8 | format_docs,
     "question": RunnablePassthrough()}
    | marketing_prompt
    | cohere_chat_model
    | output_parser)

# evaluate engineering answers
cohere_eng_results_1000_k8 = []

for question_id in rand_15:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_research"]

    answer = engineering_chain_1000_k8.invoke(question)

    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_eng_results_1000_k8.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

# evaluate marketing answers
cohere_mkt_results_1000_k8 = []

for question_id in rand_15:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_marketing"]

    answer = marketing_chain_1000_k8.invoke(question)

    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_mkt_results_1000_k8.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

# collect scores
cohere_eng_df_1000_k8 = pd.DataFrame(cohere_eng_results_1000_k8)
cohere_mkt_df_1000_k8 = pd.DataFrame(cohere_mkt_results_1000_k8)

# average scores by audience
comparison_1000_k8 = pd.DataFrame(
    [
        ["Cohere", "Engineering",
         cohere_eng_df_1000_k8["cos similarity"].mean(),
         cohere_eng_df_1000_k8["ROUGE-L F1"].mean()],

        ["Cohere", "Marketing",
         cohere_mkt_df_1000_k8["cos similarity"].mean(),
         cohere_mkt_df_1000_k8["ROUGE-L F1"].mean()]
    ],
    columns=["Model", "Team", "Avg cos similarity", "Avg ROUGE-L F1"]
)

# compare configurations
experiment_comparison = pd.concat(
    [
        baseline_comparison.assign(
            Configuration="Baseline: 128 characters, k=4"),
        comparison_1000.assign(
            Configuration="Experiment 1: 1000 characters, k=4"),
        comparison_1000_k8.assign(
            Configuration="Experiment 2: 1000 characters, k=8")
    ],
    ignore_index=True)

experiment_comparison

,Model,Team,Avg cos similarity,Avg ROUGE-L F1,Configuration
0,Cohere,Engineering,0.792634,0.138782,"Baseline: 128 characters, k=4"
1,Mistral,Engineering,0.781594,0.165601,"Baseline: 128 characters, k=4"
2,Cohere,Marketing,0.804095,0.226675,"Baseline: 128 characters, k=4"
3,Mistral,Marketing,0.806592,0.224068,"Baseline: 128 characters, k=4"
4,Cohere,Engineering,0.800110,0.138352,"Experiment 1: 1000 characters, k=4"
5,Cohere,Marketing,0.810163,0.200146,"Experiment 1: 1000 characters, k=4"
6,Cohere,Engineering,0.787757,0.135123,"Experiment 2: 1000 characters, k=8"
7,Cohere,Marketing,0.775020,0.182078,"Experiment 2: 1000 characters, k=8"


## Experiment 3: revised audience prompts


In [ ]:
# tweak engineering instructions and keep original template separate
engineering_template_exp3 = """
Answer the question for an engineering audience using the context below.
Start with a direct answer, then explain the relevant technical details.
If relevant, explain the main tradeoffs, how the details work, and an example.
Avoid unrelated background and fluffy irrelevant text.

Context: {context}
Question: {question}
"""

engineering_prompt_exp3 = ChatPromptTemplate.from_template(
    engineering_template_exp3)

# tweak marketing instructions and keep original template separate
marketing_template_exp3 = """
Answer the question for a marketing audience using the context below.
Start with a direct answer, staying high level and concise, and avoid unnecessary technical jargon.
Include the key facts and technical names needed to answer the question.
If relevant, explain the business and marketing implication.

Context: {context}
Question: {question}
"""

marketing_prompt_exp3 = ChatPromptTemplate.from_template(
    marketing_template_exp3)

# reuse experiment 1 retriever and model with revised engineering prompt
engineering_chain_exp3 = (
    {"context": retriever_1000 | format_docs,
     "question": RunnablePassthrough()}
    | engineering_prompt_exp3
    | cohere_chat_model
    | output_parser)

# reuse experiment 1 retriever and model with revised marketing prompt
marketing_chain_exp3 = (
    {"context": retriever_1000 | format_docs,
     "question": RunnablePassthrough()}
    | marketing_prompt_exp3
    | cohere_chat_model
    | output_parser)

# evaluate engineering answers
cohere_eng_results_exp3 = []

for question_id in rand_15:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_research"]

    answer = engineering_chain_exp3.invoke(question)

    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_eng_results_exp3.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

# evaluate marketing answers
cohere_mkt_results_exp3 = []

for question_id in rand_15:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_marketing"]

    answer = marketing_chain_exp3.invoke(question)

    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_mkt_results_exp3.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

# collect scores
cohere_eng_df_exp3 = pd.DataFrame(cohere_eng_results_exp3)
cohere_mkt_df_exp3 = pd.DataFrame(cohere_mkt_results_exp3)

# average scores by audience
comparison_exp3 = pd.DataFrame(
    [
        ["Cohere", "Engineering",
         cohere_eng_df_exp3["cos similarity"].mean(),
         cohere_eng_df_exp3["ROUGE-L F1"].mean()],

        ["Cohere", "Marketing",
         cohere_mkt_df_exp3["cos similarity"].mean(),
         cohere_mkt_df_exp3["ROUGE-L F1"].mean()]
    ],
    columns=["Model", "Team", "Avg cos similarity", "Avg ROUGE-L F1"]
)

# compare configurations
experiment_comparison = pd.concat(
    [
        baseline_comparison.assign(
            Configuration="Baseline: 128 characters, k=4"),
        comparison_1000.assign(
            Configuration="Experiment 1: 1000 characters, k=4"),
        comparison_1000_k8.assign(
            Configuration="Experiment 2: 1000 characters, k=8"),
        comparison_exp3.assign(
            Configuration="Experiment 3: 1000 characters, k=4, revised prompts")
    ],
    ignore_index=True)

experiment_comparison

,Model,Team,Avg cos similarity,Avg ROUGE-L F1,Configuration
0,Cohere,Engineering,0.792634,0.138782,"Baseline: 128 characters, k=4"
1,Mistral,Engineering,0.781594,0.165601,"Baseline: 128 characters, k=4"
2,Cohere,Marketing,0.804095,0.226675,"Baseline: 128 characters, k=4"
3,Mistral,Marketing,0.806592,0.224068,"Baseline: 128 characters, k=4"
4,Cohere,Engineering,0.800110,0.138352,"Experiment 1: 1000 characters, k=4"
5,Cohere,Marketing,0.810163,0.200146,"Experiment 1: 1000 characters, k=4"
6,Cohere,Engineering,0.787757,0.135123,"Experiment 2: 1000 characters, k=8"
7,Cohere,Marketing,0.775020,0.182078,"Experiment 2: 1000 characters, k=8"
8,Cohere,Engineering,0.795208,0.181813,"Experiment 3: 1000 characters, k=4, revised pr..."
9,Cohere,Marketing,0.791934,0.179975,"Experiment 3: 1000 characters, k=4, revised pr..."


## Answer and retrieval inspection

These examples are separate generations from the full validation run.


In [ ]:
# get question 0 and retrieve context
question_0 = validation_questions_answers[0]["question"]
docs_0 = retriever_1000.invoke(question_0)
context_0 = format_docs(docs_0)

inputs_0 = {"context": context_0, "question": question_0}

# get eng and mkt answers
engineering_answer_0 = (
    engineering_prompt | cohere_chat_model | output_parser
).invoke(inputs_0)

marketing_answer_0 = (
    marketing_prompt | cohere_chat_model | output_parser
).invoke(inputs_0)

print("question:", question_0)
print("\neng answer:", engineering_answer_0)
print("\nmkt answer:", marketing_answer_0)

question: What pre-trained neural retriever model is used as the non-parametric memory component in RAG, providing latent documents conditioned on the input?

eng answer: In the context of Retrieval-Augmented Generation (RAG) models, the non-parametric memory component is implemented using a **pre-trained neural retriever** that accesses a dense vector index of external knowledge, such as Wikipedia. Specifically, the retriever used in RAG is typically a **DPR (Dense Passage Retriever)** model. DPR is a neural retrieval system introduced in the paper *"Dense Passage Retrieval for Open-Domain Question Answering"* (Karpukhin et al., 2020), which encodes both queries and passages into dense vector representations using a dual-encoder architecture. 

In RAG, the DPR model serves as the retriever \( p_\eta(z|x) \), where:
- **Input \( x \)**: The query or input sequence.
- **Output \( z \)**: A set of top-K retrieved passages (latent documents) from the dense vector index of Wikipedia.

The 

In [ ]:
# show context used for question 0
print(context_0)

the input to generate the output. We marginalize the latent documents with a top-K approximation,
either on a per-output basis (assuming the same document is responsible for all tokens) or a per-token
basis (where different documents are responsible for different tokens). Like T5 [51] or BART, RAG
can be ﬁne-tuned on any seq2seq task, whereby both the generator and retriever are jointly learned.
There has been extensive previous work proposing architectures to enrich systems with non-parametric
memory which are trained from scratch for speciﬁc tasks, e.g. memory networks [64, 55], stack-
augmented networks [25] and memory layers [30]. In contrast, we explore a setting where both
parametric and non-parametric memory components are pre-trained and pre-loaded with extensive
knowledge. Crucially, by using pre-trained access mechanisms, the ability to access knowledge is
present without additional training.

The term retrieval-augmented generation (RAG) was introduced in a 2020 paper that d

In [ ]:
# show source for each retrieved chunk
for doc in docs_0:
    print(doc.metadata.get("doc_source", "Not recorded"))

ArXiv
Wikipedia
ArXiv
ArXiv


In [ ]:
# gold eng answer and cos similarity for first question
gold_ans = validation_questions_answers[0]["gold_answer_research"]

ans_embeddings = base_embeddings.embed_documents([engineering_answer_0, gold_ans])
cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

# rouge-l f1
rouge_scores = scorer.score(gold_ans, engineering_answer_0)
rougel_score = rouge_scores["rougeL"].fmeasure

print([("Cosine similarity", cosine_score),("ROUGE-L F1", rougel_score)])

[('Cosine similarity', np.float64(0.8097994777185141)), ('ROUGE-L F1', 0.18528610354223435)]


In [ ]:
# gold mkt answer and cos similarity for first question
gold_ans = validation_questions_answers[0]["gold_answer_marketing"]

ans_embeddings = base_embeddings.embed_documents([marketing_answer_0, gold_ans])
cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

# rouge-l f1
rouge_scores = scorer.score(gold_ans, marketing_answer_0)
rougel_score = rouge_scores["rougeL"].fmeasure

print([("Cosine similarity", cosine_score), ("ROUGE-L F1", rougel_score)])

[('Cosine similarity', np.float64(0.7242717160239925)), ('ROUGE-L F1', 0.22608695652173913)]


### Explaining RAG reliability


In [ ]:
# get question 134 and context
question_134 = validation_questions_answers[134]["question"]
docs_134 = retriever_1000.invoke(question_134)
context_134 = format_docs(docs_134)

inputs_134 = {"context": context_134, "question": question_134}

# eng and mkt answers
engineering_answer_134 = (engineering_prompt | cohere_chat_model | output_parser).invoke(inputs_134)
marketing_answer_134 = (marketing_prompt | cohere_chat_model | output_parser).invoke(inputs_134)

print("Question:", question_134)
print("\nEng answer:", engineering_answer_134)
print("\nMkt answer:", marketing_answer_134)

Question: What benefits does Retrieval Augmented Generation (RAG) offer over traditional language models with respect to providing reliable and up-to-date answers, and how does its approach contribute to these improvements?

Eng answer: Retrieval-Augmented Generation (RAG) significantly enhances the reliability and currency of responses generated by Large Language Models (LLMs) through a structured, two-stage process that combines information retrieval with text generation. Below is a detailed technical explanation of how RAG achieves these improvements:

### **1. Mitigation of Hallucinations and Improved Accuracy**
Traditional LLMs generate responses based solely on patterns learned during training, which can lead to "hallucinations"—fabricated or incorrect information. RAG addresses this by grounding the LLM's response in factual, retrieved data. The process involves:
- **Retriever Component**: This module queries an external knowledge base (e.g., a vector database or document corpus

In [ ]:
print(context_134)

RAG, or Retrieval Augmented Generation, is an AI technique that enhances large language models (LLMs) by allowing them to retrieve relevant information from external knowledge bases before generating a response.
How does retrieval-augmented generation improve LLM responses?
RAG mitigates LLM “hallucinations” and provides more accurate, up-to-date, and contextually relevant answers by grounding the LLM's generation in factual, retrieved data.
What are the key components of a RAG system?
A RAG system typically involves a retriever (to find relevant documents/text) and a generator (an LLM that then uses the retrieved information to form a response).
When is RAG particularly useful?
RAG is useful when LLMs need access to specialised, proprietary, or frequently updated information not present in their training data, such as company policies or recent news.
How does RAG contribute to trust in AI-generated content?

Questions often require specific context to deliver an accurate answer. Custo

In [ ]:
for doc in docs_134:
    print(doc.metadata.get("doc_source"))

WWW
WWW
ArXiv
WWW


In [ ]:
# gold eng answer and cos similarity for second question
gold_ans = validation_questions_answers[134]["gold_answer_research"]

ans_embeddings = base_embeddings.embed_documents([engineering_answer_134, gold_ans])
cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

# rouge-l f1
rouge_scores = scorer.score(gold_ans, engineering_answer_134)
rougel_score = rouge_scores["rougeL"].fmeasure

print([("Cosine similarity", cosine_score), ("ROUGE-L F1", rougel_score)])

[('Cosine similarity', np.float64(0.8902343911066452)), ('ROUGE-L F1', 0.12244897959183673)]


In [ ]:
# gold mkt answer and cos similarity for second question
gold_ans = validation_questions_answers[134]["gold_answer_marketing"]

ans_embeddings = base_embeddings.embed_documents([marketing_answer_134, gold_ans])
cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

# rouge-l f1
rouge_scores = scorer.score(gold_ans, marketing_answer_134)
rougel_score = rouge_scores["rougeL"].fmeasure

print([("Cosine similarity", cosine_score), ("ROUGE-L F1", rougel_score)])

[('Cosine similarity', np.float64(0.8487157709324491)), ('ROUGE-L F1', 0.12578616352201258)]


### Out-of-domain question


In [ ]:
# get question 225 and context
question_225 = test_questions[225]["question"]
docs_225 = retriever_1000.invoke(question_225)
context_225 = format_docs(docs_225)

inputs_225 = {"context": context_225, "question": question_225}

# eng and mkt answers
engineering_answer_225 = (
    engineering_prompt | cohere_chat_model | output_parser
).invoke(inputs_225)

marketing_answer_225 = (
    marketing_prompt | cohere_chat_model | output_parser
).invoke(inputs_225)

print("Question:", question_225)
print("\nEng answer:", engineering_answer_225)
print("\nMkt answer:", marketing_answer_225)

Question: What actor plays the role of Thanos in the Marvel Universe and the role of Cable in Deadpool 2?

Eng answer: The question you've posed is not directly related to the technical context provided, which appears to be focused on academic preprints from arXiv, primarily in the fields of computer science and machine learning, authored by researchers at DeepMind. However, I can address your question in a straightforward manner for an engineering audience, as it pertains to a factual detail rather than a technical explanation.

The actor who plays the role of **Thanos** in the Marvel Cinematic Universe (MCU) and the role of **Cable** in *Deadpool 2* is **Josh Brolin**. 

From an engineering perspective, this information could be relevant in discussions about multimedia systems, content recommendation algorithms, or natural language processing (NLP) models that handle entity recognition and linking in large datasets, such as movie databases or character-actor mappings. For example, in

In [ ]:
print(context_225)

Corresponding authors: {jordanhoﬀmann|sborgeaud|amensch|sifre}@deepmind.com
© 2023 DeepMind. All rights reserved
arXiv:2203.15556v1  [cs.CL]  29 Mar 2022

arXiv preprint arXiv:2304.09848, 2023a. URL https://arxiv.org/abs/2304.09848.
12

models. arXiv preprint arXiv:2304.06364, 2023.
9

plete answers. Thus they often lack transparency and accuracy in high stakes
arXiv:2410.15944v1  [cs.SE]  21 Oct 2024


In [ ]:
for doc in docs_225:
    print(doc.metadata.get("doc_source"))

ArXiv
ArXiv
ArXiv
ArXiv


## Selected configuration: all 75 validation questions


In [ ]:
# run all 75 questions for eng using experiment 1
cohere_eng_results_75 = []

for question_id in validation_questions_answers:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_research"]

    answer = engineering_chain_1000.invoke(question)

    # cos similarity
    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    # rouge-l f1
    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_eng_results_75.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

In [ ]:
# run all 75 questions for mkt using experiment 1
cohere_mkt_results_75 = []

for question_id in validation_questions_answers:
    question = validation_questions_answers[question_id]["question"]
    gold_ans = validation_questions_answers[question_id]["gold_answer_marketing"]

    answer = marketing_chain_1000.invoke(question)

    # cos similarity
    ans_embeddings = base_embeddings.embed_documents([answer, gold_ans])
    cosine_score = cos_sim(ans_embeddings[0], ans_embeddings[1])

    # rouge-l f1
    rouge_scores = scorer.score(gold_ans, answer)
    rougel_score = rouge_scores["rougeL"].fmeasure

    cohere_mkt_results_75.append({
        "question ID": question_id,
        "question": question,
        "generated answer": answer,
        "gold answer": gold_ans,
        "cos similarity": cosine_score,
        "ROUGE-L F1": rougel_score})

In [ ]:
# show results
cohere_eng_df_75 = pd.DataFrame(cohere_eng_results_75)
cohere_mkt_df_75 = pd.DataFrame(cohere_mkt_results_75)

# avg scores for each team
comparison_75 = pd.DataFrame(
    [
        ["Cohere", "Engineering",
         cohere_eng_df_75["cos similarity"].mean(),
         cohere_eng_df_75["ROUGE-L F1"].mean()],

        ["Cohere", "Marketing",
         cohere_mkt_df_75["cos similarity"].mean(),
         cohere_mkt_df_75["ROUGE-L F1"].mean()]
    ],
    columns=["Model", "Team", "Avg cos similarity", "Avg ROUGE-L F1"]
)

comparison_75

,Model,Team,Avg cos similarity,Avg ROUGE-L F1
0,Cohere,Engineering,0.818888,0.136126
1,Cohere,Marketing,0.813214,0.199620
